# RSNA Diz Anomaliyalarının Aşkarlanması

Bu notebook müsabiqə məlumatlarını yoxlayır, MRI şəkillərindən xüsusiyyətlər çıxarır, modeli qiymətləndirir və Kaggle üçün `submission.csv` yaradır.

Submission faylındakı `StudyInstanceUID` dəyərləri sample submission faylındakı ID-lərlə dəqiq uyğunlaşdırılır.

In [5]:
import argparse
from pathlib import Path
from typing import Mapping, Sequence

import numpy as np
import pandas as pd

LABEL_COLS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

def load_competition_tables(train_path, test_path, train_series_path, test_series_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    train_series_df = pd.read_csv(train_series_path)
    test_series_df = pd.read_csv(test_series_path)
    return train_df, test_df, train_series_df, test_series_df

def audit_null_values(dataframe, label_cols, report_col="Report"):
    columns_to_check = [column for column in [report_col, *label_cols] if column in dataframe.columns]
    if not columns_to_check:
        raise ValueError("None of the requested report or label columns exist in the data")

    summary = pd.DataFrame(index=columns_to_check)
    summary["null_count"] = dataframe[columns_to_check].isna().sum()
    summary["null_percent"] = (summary["null_count"] / max(len(dataframe), 1) * 100).round(2)
    summary["blank_report_count"] = 0

    if report_col in dataframe.columns:
        report_text = dataframe[report_col].astype("string")
        blank_report = report_text.notna() & report_text.str.strip().eq("").fillna(False)
        summary.loc[report_col, "blank_report_count"] = int(blank_report.sum())
        summary.loc[report_col, "null_percent"] = (
            (summary.loc[report_col, "null_count"] + blank_report.sum()) / max(len(dataframe), 1) * 100
        ).round(2)

    return summary

def prepare_labeled_studies(train_df, label_cols):
    missing_columns = [column for column in label_cols if column not in train_df.columns]
    if missing_columns:
        raise ValueError(f"Training data is missing label columns: {missing_columns}")
    return train_df.dropna(subset=list(label_cols)).copy()

def prepare_report_training_data(train_df, label_cols, report_col="Report"):
    if report_col not in train_df.columns:
        raise ValueError(f"Training data does not contain the required {report_col!r} column")

    labeled = prepare_labeled_studies(train_df, label_cols)
    report_text = labeled[report_col].astype("string")
    has_report = report_text.notna() & report_text.str.strip().ne("").fillna(False)
    return labeled.loc[has_report].copy()

def audit_training_modalities(train_df, train_series_df, label_cols, report_col="Report"):
    required_columns = {"StudyInstanceUID"}
    if not required_columns.issubset(train_df.columns) or not required_columns.issubset(train_series_df.columns):
        raise ValueError("Training and series tables must contain StudyInstanceUID")

    labeled = prepare_labeled_studies(train_df, label_cols)
    series_study_ids = set(train_series_df["StudyInstanceUID"].dropna().astype(str))
    has_series = labeled["StudyInstanceUID"].astype(str).isin(series_study_ids)

    if report_col in labeled.columns:
        report_text = labeled[report_col].astype("string")
        has_report = report_text.notna() & report_text.str.strip().ne("").fillna(False)
    else:
        has_report = pd.Series(False, index=labeled.index)

    return pd.DataFrame(
        {
            "study_count": [
                len(train_df),
                len(labeled),
                int((has_report & has_series).sum()),
                int((~has_report & has_series).sum()),
                int((has_report & ~has_series).sum()),
                int((~has_report & ~has_series).sum()),
            ]
        },
        index=[
            "all_training_studies",
            "studies_with_all_labels",
            "labeled_with_report_and_series",
            "labeled_missing_report_with_series",
            "labeled_with_report_missing_series",
            "labeled_missing_report_and_series",
        ],
    )

def prepare_image_training_data(train_df, train_series_df, label_cols):
    labeled = prepare_labeled_studies(train_df, label_cols)
    series_study_ids = set(train_series_df["StudyInstanceUID"].dropna().astype(str))
    has_series = labeled["StudyInstanceUID"].astype(str).isin(series_study_ids)
    return labeled.loc[has_series].copy()

def normalize_prediction_frame(predictions, label_cols):
    if isinstance(predictions, Mapping):
        frame = pd.DataFrame.from_dict(predictions, orient="index", columns=list(label_cols))
        return frame

    if hasattr(predictions, "to_pandas"):
        predictions = predictions.to_pandas()

    if isinstance(predictions, pd.DataFrame):
        frame = predictions.copy()
    elif isinstance(predictions, str):
        frame = pd.read_csv(predictions, index_col=0)
    else:
        raise TypeError("Unsupported predictions object.")

    if list(frame.columns) != list(label_cols):
        missing = [c for c in label_cols if c not in frame.columns]
        if missing:
            raise ValueError(f"Prediction frame is missing required columns: {missing}")
        extra = [c for c in frame.columns if c not in label_cols]
        if extra:
            raise ValueError(f"Prediction frame contains unexpected columns: {extra}")
        frame = frame[list(label_cols)]

    frame.index = frame.index.astype(str)
    return frame

def generate_submission(predictions, sample_submission_path, output_path, label_cols=None):
    sample_submission = pd.read_csv(sample_submission_path)
    if "StudyInstanceUID" not in sample_submission.columns:
        raise ValueError("Sample submission is missing StudyInstanceUID")

    if label_cols is None:
        label_cols = [column for column in sample_submission.columns if column != "StudyInstanceUID"]

    prediction_frame = normalize_prediction_frame(predictions, label_cols)
    submission_ids = sample_submission["StudyInstanceUID"].astype(str)

    prediction_index = prediction_frame.index.astype(str)
    if set(submission_ids) != set(prediction_index):
        missing = sorted(set(submission_ids) - set(prediction_index))
        unexpected = sorted(set(prediction_index) - set(submission_ids))
        raise ValueError(
            "Prediction IDs do not match the sample submission IDs. "
            f"Missing: {missing[:5]}; Unexpected: {unexpected[:5]}"
        )

    aligned = prediction_frame.reindex(submission_ids.tolist())
    values = aligned.to_numpy(dtype=np.float32)

    if not np.isfinite(values).all():
        raise ValueError("Predictions contain NaN or infinite values")
    if ((values < 0) | (values > 1)).any():
        raise ValueError("Predicted probabilities must be between 0 and 1")

    submission = sample_submission.copy()
    submission.loc[:, list(label_cols)] = values

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(output, index=False)

    saved = pd.read_csv(output)
    if saved.shape != submission.shape or saved.columns.tolist() != submission.columns.tolist():
        raise ValueError("Saved submission does not match the expected sample submission schema")

    print(f"Saved {len(saved)} predictions to {output}")
    return saved

In [6]:
COMPETITION_DATA_DIR = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)
REQUIRED_COMPETITION_FILES = (
    "train.csv",
    "test.csv",
    "train_series.csv",
    "test_series.csv",
    "sample_submission.csv",
)


def find_competition_data_dir():
    missing_files = [
        filename
        for filename in REQUIRED_COMPETITION_FILES
        if not (COMPETITION_DATA_DIR / filename).is_file()
    ]
    if missing_files:
        raise FileNotFoundError(
            f"Competition input is not mounted at {COMPETITION_DATA_DIR}. "
            f"Missing files: {missing_files}. In Kaggle, use Add Input to attach "
            "RSNA Knee Abnormality Detection, then restart and run the notebook."
        )

    return COMPETITION_DATA_DIR

In [7]:
# Find the competition data folder automatically, even if Kaggle mounts it under a different name.
COMPETITION_DATA_DIR = find_competition_data_dir()
print(f"Competition data found at: {COMPETITION_DATA_DIR}")

train_df, test_df, train_series_df, test_series_df = load_competition_tables(
    train_path=COMPETITION_DATA_DIR / "train.csv",
    test_path=COMPETITION_DATA_DIR / "test.csv",
    train_series_path=COMPETITION_DATA_DIR / "train_series.csv",
    test_series_path=COMPETITION_DATA_DIR / "test_series.csv",
)

null_summary = audit_null_values(train_df, LABEL_COLS)
display(null_summary)

labeled = prepare_labeled_studies(train_df, LABEL_COLS)
print(f"Total training rows: {len(train_df)}")
print(f"Rows with all 12 labels: {len(labeled)}")

modality_summary = audit_training_modalities(train_df, train_series_df, LABEL_COLS)
display(modality_summary)

image_labeled = prepare_image_training_data(train_df, train_series_df, LABEL_COLS)
print(f"Label-complete studies with a listed image series: {len(image_labeled)}")

if "Report" in train_df.columns:
    report_labeled = prepare_report_training_data(train_df, LABEL_COLS)
    print(f"Rows with complete labels and nonblank reports: {len(report_labeled)}")
else:
    print("No Report column found; report-based training data was not prepared.")

FileNotFoundError: Competition input is not mounted at /kaggle/input/competitions/rsna-knee-abnormality-detection. Missing files: ['train.csv', 'test.csv', 'train_series.csv', 'test_series.csv', 'sample_submission.csv']. In Kaggle, use Add Input to attach RSNA Knee Abnormality Detection, then restart and run the notebook.

In [ ]:
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("Report available in test:", "Report" in test_df.columns)
print("Train rows:", len(train_df), "Test rows:", len(test_df))
print("Train series rows:", len(train_series_df), "Test series rows:", len(test_series_df))

print("\nMissing-value summary:")
display(null_summary)

print("\nReport and image-series coverage:")
display(modality_summary)

## MRI modeli və proqnoz

Bu hüceyrə etiketləri tam olan MRI study-lərindən xüsusiyyətlər çıxarır, hər label üçün 5-fold yoxlama aparır və sonra bütün uyğun study-lərlə modeli yenidən öyrədir. Vaxtı azaltmaq üçün hər series-dən maksimum 16 bərabər aralıqlı DICOM şəkli istifadə olunur.

Kaggle-da GPU-nu aktiv et və DINOv2 modelini notebook-a Input kimi əlavə et. Hüceyrə uğurla tamamlananda 3 test study üçün proqnozlar yaradılır; növbəti hüceyrə onları submission faylına yazır.

In [ ]:
from pathlib import Path

import pydicom
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoImageProcessor, AutoModel

DATA_ROOT = COMPETITION_DATA_DIR
TRAIN_IMAGE_ROOT = DATA_ROOT / "train_series"
TEST_IMAGE_ROOT = DATA_ROOT / "test_series"
DINOV2_PATH = Path("/kaggle/input/models/metaresearch/dinov2/pytorch/large/1")
MAX_SLICES_PER_SERIES = 16

if not DINOV2_PATH.exists():
    raise FileNotFoundError(
        f"DINOv2 model not found at {DINOV2_PATH}. Add the DINOv2 model as a Kaggle input."
    )

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running MRI feature extraction.")

device = torch.device("cuda")
processor = AutoImageProcessor.from_pretrained(str(DINOV2_PATH))
image_encoder = AutoModel.from_pretrained(str(DINOV2_PATH)).to(device).eval()


def extract_study_feature(study_id, series_table, image_root):
    study_rows = series_table[
        series_table["StudyInstanceUID"].astype(str) == str(study_id)
    ]
    if study_rows.empty:
        raise ValueError(f"No series metadata found for study {study_id}")

    series_features = []
    for _, series_row in study_rows.iterrows():
        series_id = str(series_row["SeriesInstanceUID"])
        series_path = image_root / str(study_id) / series_id
        if not series_path.is_dir():
            raise FileNotFoundError(f"DICOM series directory not found: {series_path}")

        dicom_files = sorted(
            path for path in series_path.iterdir()
            if path.is_file() and path.suffix.lower() == ".dcm"
        )
        if not dicom_files:
            raise ValueError(f"No DICOM files found in {series_path}")

        if len(dicom_files) > MAX_SLICES_PER_SERIES:
            selected = np.linspace(
                0, len(dicom_files) - 1, MAX_SLICES_PER_SERIES, dtype=int
            )
            dicom_files = [dicom_files[index] for index in selected]

        slice_features = []
        for dicom_path in dicom_files:
            dicom = pydicom.dcmread(str(dicom_path))
            pixels = np.squeeze(dicom.pixel_array).astype(np.float32)
            if pixels.ndim != 2:
                raise ValueError(f"Expected a 2D MRI slice in {dicom_path}, got {pixels.shape}")

            pixel_min = float(pixels.min())
            pixel_range = float(pixels.max() - pixel_min)
            if pixel_range > 0:
                pixels = (pixels - pixel_min) / pixel_range
            else:
                pixels = np.zeros_like(pixels)

            image = Image.fromarray((pixels * 255).astype(np.uint8)).convert("RGB")
            model_inputs = processor(images=image, return_tensors="pt").to(device)
            with torch.inference_mode():
                output = image_encoder(**model_inputs)
            slice_features.append(output.last_hidden_state[0, 0].float().cpu())

        series_features.append(torch.stack(slice_features).mean(dim=0))

    return torch.stack(series_features).mean(dim=0).numpy()


labeled_image_studies = prepare_image_training_data(
    train_df, train_series_df, LABEL_COLS
).reset_index(drop=True)
train_ids = labeled_image_studies["StudyInstanceUID"].astype(str).tolist()
test_ids = test_df["StudyInstanceUID"].astype(str).tolist()

if len(set(train_ids)) != len(train_ids) or len(set(test_ids)) != len(test_ids):
    raise ValueError("StudyInstanceUID values must be unique in train and test tables")

train_features = []
for index, study_id in enumerate(train_ids, start=1):
    train_features.append(
        extract_study_feature(study_id, train_series_df, TRAIN_IMAGE_ROOT)
    )
    if index % 10 == 0 or index == len(train_ids):
        print(f"Extracted training MRI features: {index}/{len(train_ids)}")

test_features = []
for index, study_id in enumerate(test_ids, start=1):
    test_features.append(
        extract_study_feature(study_id, test_series_df, TEST_IMAGE_ROOT)
    )
    print(f"Extracted test MRI features: {index}/{len(test_ids)}")

X_train_all = np.stack(train_features).astype(np.float32)
X_test = np.stack(test_features).astype(np.float32)
y_train_all = labeled_image_studies[LABEL_COLS].to_numpy(dtype=np.int32)

if not np.isin(y_train_all, [0, 1]).all():
    raise ValueError("Training labels must contain only 0 and 1")

validation_probabilities = np.zeros_like(y_train_all, dtype=np.float32)
test_probabilities_by_label = {}

for label_index, label in enumerate(LABEL_COLS):
    label_values = y_train_all[:, label_index]
    class_counts = np.bincount(label_values, minlength=2)
    folds = min(5, int(class_counts.min()))

    if folds < 2:
        prior = float((label_values.sum() + 1) / (len(label_values) + 2))
        validation_probabilities[:, label_index] = prior
        test_probabilities_by_label[label] = np.full(len(X_test), prior, dtype=np.float32)
        print(
            f"{label}: fewer than 2 examples in one class; "
            f"using smoothed prevalence {prior:.3f} instead of cross-validation."
        )
        continue

    splitter = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    for fit_indices, valid_indices in splitter.split(X_train_all, label_values):
        fold_model = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000),
        )
        fold_model.fit(X_train_all[fit_indices], label_values[fit_indices])
        validation_probabilities[valid_indices, label_index] = fold_model.predict_proba(
            X_train_all[valid_indices]
        )[:, 1]

    try:
        auc = roc_auc_score(label_values, validation_probabilities[:, label_index])
    except ValueError:
        auc = float("nan")
    f1 = f1_score(
        label_values,
        validation_probabilities[:, label_index] >= 0.5,
        zero_division=0,
    )
    print(f"{label}: out-of-fold AUC={auc:.3f}, F1={f1:.3f}, folds={folds}")

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000),
    )
    final_model.fit(X_train_all, label_values)
    test_probabilities_by_label[label] = final_model.predict_proba(X_test)[:, 1]

predicted_values = np.column_stack(
    [test_probabilities_by_label[label] for label in LABEL_COLS]
)
test_probabilities = pd.DataFrame(
    predicted_values,
    index=test_ids,
    columns=LABEL_COLS,
)
print(f"Generated MRI predictions for {len(test_probabilities)} test studies.")

In [ ]:
validation_rows = []
for label_index, label in enumerate(LABEL_COLS):
    label_values = y_train_all[:, label_index]
    probabilities = validation_probabilities[:, label_index]
    try:
        auc = roc_auc_score(label_values, probabilities)
    except ValueError:
        auc = float("nan")

    validation_rows.append(
        {
            "Label": label,
            "Positive studies": int(label_values.sum()),
            "Negative studies": int(len(label_values) - label_values.sum()),
            "OOF ROC-AUC": auc,
            "OOF F1 at 0.5": f1_score(
                label_values,
                probabilities >= 0.5,
                zero_division=0,
            ),
        }
    )

validation_results = pd.DataFrame(validation_rows).set_index("Label")
display(validation_results.round(3))
print(f"Validation studies: {len(y_train_all)}")
print(f"Test studies with MRI predictions: {len(test_probabilities)}")

## Qiymətləndirmə nəticələri

Yuxarıdakı cədvəl 58 etiketli training study üzərində 5 hissəli növbəli yoxlamanın nəticələrini göstərir. ROC-AUC modelin müsbət halları mənfi hallardan nə dərəcədə yaxşı sıraladığını, F1 isə 0.5 ehtimal həddində nəticəni göstərir. Məlumat az olduğuna görə nəticələr ilkin xarakter daşıyır. Bu sınaqda Medial OA daha yaxşı nəticə göstərib; ACL və Lateral Meniscus nəticələri təsadüfi təxminə yaxındır, MCL üzrə F1 isə aşağıdır.

MRI xüsusiyyətləri bütün 3 test study üçün çıxarılıb. Növbəti hüceyrə proqnozları `/kaggle/working/submission.csv` faylına yazır. Faylın yaranması onun formatının hazır olduğunu göstərir; müsabiqə balı isə Kaggle qiymətləndirdikdən sonra məlum olacaq.

## Kaggle submission faylının yaradılması

Bu hüceyrə MRI modelinin proqnozlarını `/kaggle/working/submission.csv` faylına yazır. Bunun üçün əvvəlki MRI proqnozu hüceyrəsi xətasız tamamlanmalıdır.

In [ ]:
sample_path = COMPETITION_DATA_DIR / "sample_submission.csv"
output_path = "/kaggle/working/submission.csv"

if "test_probabilities" not in globals():
    raise RuntimeError("Run the MRI feature extraction and prediction cell before creating the submission.")

submission = generate_submission(
    predictions=test_probabilities,
    sample_submission_path=sample_path,
    output_path=output_path,
    label_cols=LABEL_COLS,
)

print(f"Submission saved to: {output_path}")
submission.head()